# Harga setelah Lebaran: turun atau lengket?

**Abstrak ringkas.** Klaim umum menyatakan harga pangan "naik cepat, turun lambat" sesudah Ramadan. Edisi ini menguji klaim tersebut dengan definisi kelengketan harga yang eksplisit pada delapan musim Ramadan (1440-1447 H) dan sepuluh komoditas pangan. Hasilnya menolak generalisasi: kelengketan pasca-Lebaran sangat bergantung pada rezim pembentukan harga. Kenaikan pada komoditas hortikultura bergejolak nyaris hilang seluruhnya, sedangkan kenaikan pada komoditas berharga administratif-standar cenderung menetap.

In [ ]:
import sys
from pathlib import Path

import json
import numpy as np
import pandas as pd

from tools import gaya

gaya.terapkan()

df = pd.read_csv("data/harga-lengket.csv")
uji = df[df.diuji].copy()
print("baris total:", len(df), "| diuji:", len(uji))
print("musim dalam tabel:", sorted(df.tahun_hijri.unique()))
print("gugur (dokumentasi): musim 1442 H tak lolos syarat >= 6 snapshot pada jendela Ramadan")

baris total: 70 | diuji: 51
musim dalam tabel: [np.int64(1440), np.int64(1441), np.int64(1443), np.int64(1444), np.int64(1445), np.int64(1446), np.int64(1447)]
gugur (dokumentasi): musim 1442 H tak lolos syarat >= 6 snapshot pada jendela Ramadan


## Latar Belakang

Ada dua tesis populer yang bertentangan mengenai harga pasca-Lebaran. Tesis pertama: harga kembali turun setelah momentum Ramadan berakhir. Tesis kedua (roket-dan-bulu dalam literatur harga): harga yang terlanjur naik enggan turun. Edisi ini merumuskan pertanyaannya secara terukur: *dari bagian kenaikan harga musim Ramadan, berapa besar yang masih menetap 30 dan 42 hari setelah Lebaran?*

## Data & Metode

Sumber tunggal [1]: PIHPS (Pusat Informasi Harga Pangan Strategis, bi.go.id), rerata nasional pasar tradisional mingguan (snapshot Rabu) untuk sepuluh komoditas, delapan musim Ramadan 1440-1447 H, berbasis panen Edisi 008 yang berhenti pada 29 April 2026 [2]. Kalender Islam memakai tanggal ketetapan pemerintah yang terverifikasi di proyek [3]. Definisi metrik:

- **basis**: rerata harga minggu-minggu 70 s.d. 35 hari sebelum 1 Ramadan.
- **kenaikan puncak**: maksimum harga relatif basis pada jendela H-35 s.d. H+8 relatif Lebaran. Musim dengan kenaikan di bawah 2% tidak diuji (tidak ada kenaikan untuk diamati kelengketannya).
- **sisa**: harga relatif basis pada snapshot terdekat +30 dan +42 hari pasca-Lebaran (toleransi 4 hari).
- **indeks lengket**: (sisa - 1) / kenaikan puncak. Nilai 0 : pulih penuh; 1 : bertahan penuh; > 1 : harga justru naik setelah Lebaran; < 0 : turun melewati basis.

Statistik ringkas memakai median antar-musim, dengan selang kepercayaan 95% dari bootstrap 10.000 pengambilan ulang atas musim (benih acak 2026, direproduksikan persis). Syarat kenyaringan: basis tersedia dan jendela Ramadan berisi sedikitnya enam snapshot; satu musim (1442 H) gugur serentak pada semua komoditas karena celah rilis sumber (dokumentasi [2]).

In [2]:
# verifikasi definisi & ukuran sampel
assert len(df) == 70 and len(uji) == 51
assert set(df.tahun_hijri) == {1440, 1441, 1443, 1444, 1445, 1446, 1447}

def bootstrap_median(v, n=10_000, seed=2026):
    rng = np.random.default_rng(seed)
    v = np.asarray(v, dtype=float)
    med = np.empty(n)
    for i in range(n):
        med[i] = np.median(rng.choice(v, size=len(v), replace=True))
    return float(np.median(v)), float(np.percentile(med, 2.5)), float(np.percentile(med, 97.5))

tabel = {}
for kom, g in uji.groupby("komoditas"):
    med, lo, hi = bootstrap_median(g["lengket30"].dropna())
    tabel[kom] = {"n_musim": int(g.lengket30.notna().sum()),
                  "lengket30_med": med, "lo": lo, "hi": hi,
                  "lengket42_med": float(np.nanmedian(g["lengket42"]))}
for kom, t in sorted(tabel.items(), key=lambda kv: -kv[1]["lengket30_med"]) if True else tabel.items():
    print(f"{kom:14s} n={t['n_musim']}  lengket@+30: {t['lengket30_med']:+.2f} "
          f"[{t['lo']:+.2f}, {t['hi']:+.2f}]  |  @+42: {t['lengket42_med']:+.2f}")

Minyak Goreng  n=1  lengket@+30: +2.20 [+2.20, +2.20]  |  @+42: +2.46
Gula Pasir     n=5  lengket@+30: +1.00 [-0.18, +1.88]  |  @+42: +1.25
Bawang Putih   n=4  lengket@+30: +0.80 [+0.27, +1.93]  |  @+42: +0.95
Beras          n=2  lengket@+30: +0.76 [+0.52, +1.00]  |  @+42: +0.79
Daging Ayam    n=5  lengket@+30: +0.60 [-0.61, +1.49]  |  @+42: +0.76
Bawang Merah   n=4  lengket@+30: +0.52 [+0.36, +0.88]  |  @+42: +0.34
Telur Ayam     n=4  lengket@+30: +0.44 [-0.36, +3.32]  |  @+42: +0.40
Daging Sapi    n=6  lengket@+30: +0.41 [+0.18, +0.60]  |  @+42: +0.41
Cabai Merah    n=5  lengket@+30: +0.13 [-0.15, +1.39]  |  @+42: +0.18
Cabai Rawit    n=5  lengket@+30: -0.28 [-1.08, +1.35]  |  @+42: -0.20


## Hasil

Gambar pertama memetakan setiap musim yang diuji (51 pasangan komoditas x musim) ke bidang; kenaikan sebagai sumbu mendatar, sisanya pada H+30 sebagai sumbu tegak. Titik pada garis diagonal adalah kelengketan penuh; titik pada garis nol adalah pemulihan penuh.

In [3]:
def grafik_1(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Turun atau lengket, terlihat dari 51 musim pasca-Lebaran",
        "Setiap titik: satu komoditas pada satu musim. Dekat garis nol berarti pulih; "
        "di diagonal (berlabel) berarti kenaikan bertahan penuh pada H+30.",
        "PIHPS via arsip Edisi 008; definisi indeks lengket di Metode",
        int(len(uji.dropna(subset=["rel_sisa30"]))),
    )
    x = (uji["rel_puncak"] - 1) * 100
    y = (uji["rel_sisa30"] - 1) * 100
    terpetakan = uji.dropna(subset=["rel_sisa30"])
    ax.scatter(x, y, s=26, color=gaya.INK2, alpha=0.45, zorder=3,
               label=f"musim diuji (n={len(terpetakan)})")
    med = uji.groupby("komoditas").agg(mx=("rel_puncak", "median"), my=("rel_sisa30", "median"))
    ax.scatter((med.mx - 1) * 100, (med.my - 1) * 100, s=90, marker="D",
               color=gaya.AKSEN, zorder=4, label="median per komoditas")
    ax.plot([0, 40], [0, 40], color=gaya.INK2, ls="--", lw=1.0, zorder=2)
    ax.text(30.5, 33.5, "diagonal = lengket penuh", fontsize=8.2 * fs,
            color=gaya.INK2, rotation=26)
    ax.axhline(0, color=gaya.INK2, lw=0.9, ls=":", zorder=2)
    ax.text(34.5, -3.5, "garis nol = pulih penuh", fontsize=8.2 * fs, color=gaya.INK2)
    label_pos = {"Cabai Rawit": None, "Cabai Merah": None, "Gula Pasir": None,
                 "Daging Ayam": None, "Minyak Goreng": None, "Bawang Putih": None,
                 "Bawang Merah": None, "Telur Ayam": None, "Daging Sapi": None}
    off = {"Cabai Rawit": (1, -9, "left"), "Cabai Merah": (1, -9, "left"),
           "Gula Pasir": (1, 10, "left"), "Minyak Goreng": (-1, 10, "right"),
           "Bawang Putih": (1, -9, "left"), "Bawang Merah": (1, -9, "left"),
           "Telur Ayam": (-1, 10, "right"), "Daging Ayam": (1, -16, "left"),
           "Daging Sapi": (-1, -12, "right"), "Beras": (1, -8, "left")}
    for kom in off:
        if kom in med.index:
            dx, dy, hami = off[kom]
            ax.annotate(kom, ((med.loc[kom, "mx"] - 1) * 100, (med.loc[kom, "my"] - 1) * 100),
                        textcoords="offset points", xytext=(dx * 7, dy), fontsize=8.2 * fs,
                        color=gaya.INK, ha=hami)
    ax.set_xlabel("kenaikan puncak musim Ramadan (% di atas basis)", fontsize=9.5 * fs)
    ax.set_ylabel("sisa kenaikan pada H+30 (% di atas basis)", fontsize=9.5 * fs)
    ax.set_xlim(-2, 42)
    ax.set_ylim(-14, 42)
    ax.legend(fontsize=8.6 * fs, loc="upper left", frameon=False)
    ax.grid(True)
    gaya.simpan(fig, "01-turun-atau-lengket", mode)


for mode in gaya.MODE:
    grafik_1(mode)

Gambar kedua meringkas median indeks kelengketan per komoditas beserta selang kepercayaan 95% (bootstrap 10.000 kali atas musim, benih 2026). Komoditas dengan kurang dari empat musim yang diuji diberi tanda pemisah.

In [4]:
def grafik_2(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Kelengketan median: hortikultura pulih, komoditas standar menetap",
        "Median indeks lengket +H30 dan selang 95%-nya. Nilai 0 = pulih penuh; "
        "1 = bertahan penuh; batang pudar menandai musim diuji kurang dari 4.",
        "PIHPS via arsip Edisi 008; bootstrap 10.000 ulang, benih 2026",
        int(len(uji)),
    )
    urut = sorted(tabel.items(), key=lambda kv: kv[1]["lengket30_med"])
    nama = [k for k, _ in urut]
    ys = np.arange(len(nama))
    for i, (kom, t) in enumerate(urut):
        kecil = t["n_musim"] < 4
        warna = gaya.INK2 if kecil else gaya.INK
        ax.barh(i, t["lengket30_med"], color=warna, alpha=0.9 if not kecil else 0.55,
                height=0.6, zorder=3)
        if t["n_musim"] >= 2:
            ax.plot([t["lo"], t["hi"]], [i, i], color=gaya.AKSEN, lw=2.0, zorder=4)
            ax.plot([t["lo"], t["hi"]], [i, i], "|", color=gaya.AKSEN, ms=6, zorder=4)
        ax.text(max(t["lengket30_med"], t["hi"] if t["n_musim"] >= 2 else 0) + 0.09, i,
                f"n={t['n_musim']}", va="center", fontsize=8 * fs, color=gaya.INK2)
    ax.axvline(0, color=gaya.INK2, lw=0.9)
    ax.axvline(1, color=gaya.AKSEN, lw=1.1, ls="--")
    ax.text(1.02, -0.75, "1 = lengket penuh", fontsize=8.2 * fs, color=gaya.AKSEN)
    ax.text(-0.02, -0.75, "0 = pulih penuh", fontsize=8.2 * fs, color=gaya.INK2, ha="right")
    ax.set_yticks(ys)
    ax.set_yticklabels(nama, fontsize=9 * fs)
    ax.set_xlim(-0.6, 3.25)
    ax.set_xlabel("median indeks lengket +H30 ((sisa-1)/(puncak-1))", fontsize=9.5 * fs)
    ax.grid(True, axis="x"); ax.grid(False, axis="y")
    gaya.simpan(fig, "02-indeks-lengket", mode)


for mode in gaya.MODE:
    grafik_2(mode)

## Pembahasan

Temuan utama dua lapis. Pertama, penolakan atas tesis roket-dan-bulu yang generalis: untuk seluruh komponen hortikultura yang naik sungguhan di musim Ramadan, kenaikan itu hilang hampir seluruhnya dalam 30 hari — cabai rawit terturun tajam (median lengket -0,28: malah sepatutnya di bawah basis), cabai merah 0,13, daging sapi 0,41, telur 0,44, bawang merah 0,52, daging ayam 0,60. Kedua, ketika kenaikan terjadi pada komoditas dengan pembentukan harga administratif-standar (gula pasir; dan anomali minyak goreng musim krisis 2022, satu-satunya musimnya yang teruji), kenaikannya cenderung menetap: gula bertahan penuh (median 1,00, dan medan +H42 malah 1,25), minyak 2,20 sebagai sinyal rezim krisis, bukan musiman.

Konsistensi antar titik ukur H+30 dan H+42 (kolom kanan tabel metrik) menunjukkan pola ini bukan pemilihan waktu potret yang kebetulan.

In [5]:
metrik = {
    "definisi": {"basis": "rerata minggu -70 s.d. -35 relatif 1 Ramadan",
                 "jendela_puncak": "hari -35 s.d. +8 relatif Lebaran (ketum pemerintah)",
                 "ambang_diuji_pct": 2.0, "toleransi_snap_hari": 4},
    "n": {"baris_total": len(df), "diuji": len(uji),
          "musim_gugur": ["1442 H (celah rilis sumber; <6 snapshot jendela Ramadan)"]},
    "per_komoditas": {kom: {"n_musim": t["n_musim"],
                            "lengket30_med": round(t["lengket30_med"], 3),
                            "ci95": [round(t["lo"], 3), round(t["hi"], 3)],
                            "lengket42_med": round(t["lengket42_med"], 3)}
                      for kom, t in tabel.items()},
    "bootstrap": {"ulang": 10_000, "benih": 2026, "estimand": "median antar-musim"},
}
print(json.dumps(metrik, indent=2, ensure_ascii=False))
Path("data/metrik-011.json").write_text(json.dumps(metrik, indent=2, ensure_ascii=False))

{
  "definisi": {
    "basis": "rerata minggu -70 s.d. -35 relatif 1 Ramadan",
    "jendela_puncak": "hari -35 s.d. +8 relatif Lebaran (ketum pemerintah)",
    "ambang_diuji_pct": 2.0,
    "toleransi_snap_hari": 4
  },
  "n": {
    "baris_total": 70,
    "diuji": 51,
    "musim_gugur": [
      "1442 H (celah rilis sumber; <6 snapshot jendela Ramadan)"
    ]
  },
  "per_komoditas": {
    "Bawang Merah": {
      "n_musim": 4,
      "lengket30_med": 0.516,
      "ci95": [
        0.362,
        0.881
      ],
      "lengket42_med": 0.336
    },
    "Bawang Putih": {
      "n_musim": 4,
      "lengket30_med": 0.797,
      "ci95": [
        0.267,
        1.933
      ],
      "lengket42_med": 0.953
    },
    "Beras": {
      "n_musim": 2,
      "lengket30_med": 0.758,
      "ci95": [
        0.515,
        1.0
      ],
      "lengket42_med": 0.793
    },
    "Cabai Merah": {
      "n_musim": 5,
      "lengket30_med": 0.13,
      "ci95": [
        -0.15,
        1.387
      ],
      "lengke

2089

## Batasan & cara reproduksi

Batasan utama. (1) Metrik menilai harga agregat mingguan nasional pasar tradisional; pembeli per eceran dan wilayah yang berbeda dapat mengenyam lintasan lain. (2) Kelengketan adalah sifat kenaikan, sehingga musim tanpa kenaikan bermakna dikeluarkan dari uji — dan bagi komoditas stabil, sensitif pada ambang 2%. (3) Delapan musim adalah ukuran sampel kecil untuk menguji interaksi kalender; penduga pada tepi distribusi dilaporkan jujur lebar. (4) Tahun 2019/2020 membawa guncangan harga besar (kenaikan 1440 H cabai-bawang dan krisis minyak goreng 2022) yang berada di luar variabilitas biasa; temuan diringkas median untuk alasan itu. (5) Tanggal Ramadan dan Lebaran memakai penetapan pemerintah [3]; kalender lain bergeser satu-dua hari. (6) Panen memakai snapshot Rabu; titik +30/+42 adalah snapshot terdekat dalam toleransi empat hari.